# TFM — Feature Engineering (H2)

> **Objetivo:** construir el dataset de modelado ('df_modelo') a partir de las tablas OULAD.
> Ventana de predicción: días 0–27. Población: estudiantes activos en día 27 (n=27.553).
>
> **Estructura:**
> 0. Setup y carga de datos
> 1. Features de VLE (comportamiento en plataforma)
> 2. Features de Assessment (evaluaciones en ventana)
> 3. Ensamblado: 'df_modelo'
> 4. Exportar

## 0. Setup y carga de datos

In [1]:
import pandas as pd
import numpy as np

PATH = "../datos/OULAD/"

courses        = pd.read_csv(PATH + "courses.csv")
assessments    = pd.read_csv(PATH + "assessments.csv")
vle            = pd.read_csv(PATH + "vle.csv")
student_info   = pd.read_csv(PATH + "studentInfo.csv")
student_reg    = pd.read_csv(PATH + "studentRegistration.csv")
student_assess = pd.read_csv(PATH + "studentAssessment.csv")
student_vle    = pd.read_csv(PATH + "studentVle.csv")

print("Tablas cargadas:", ["courses","assessments","vle","student_info","student_reg","student_assess","student_vle"])

Tablas cargadas: ['courses', 'assessments', 'vle', 'student_info', 'student_reg', 'student_assess', 'student_vle']


**Comentarios:**

Construimos un dataset autocontenido para evitar dependencias de otros archivos, por ese motivo aquí volvemos a cargar de nuevo las 7 tablas de OULAD y creamos las variables intermedias necesarias.

### Reconstruir población base y variables derivadas del EDA

Recreamos 'df_activos', 'assess_ventana' y 'entregaron' para no depender del estado del kernel de '01_EDA.ipynb'.

In [2]:
# Variable objetivo
student_info["riesgo"] = student_info["final_result"].isin(["Fail", "Withdrawn"]).astype(int)

# Corrección imd_band
student_info["imd_band"] = student_info["imd_band"].replace("10-20", "10-20%")
student_info["imd_band"] = student_info["imd_band"].fillna("Missing")

# Filtro de activos: excluir date_unregistration <= 27
df = student_info.merge(
    student_reg[["id_student", "code_module", "code_presentation", "date_unregistration"]],
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)
df_activos = df[
    df["date_unregistration"].isna() | (df["date_unregistration"] > 27)
].copy()

print(f"Estudiantes activos en día 27: {len(df_activos):,}")
print(f"Balance de clases — riesgo=1: {df_activos['riesgo'].mean():.3f}")

Estudiantes activos en día 27: 27,553
Balance de clases — riesgo=1: 0.442


**Comentarios:**

Recuperamos las 3 transformaciones documentadas en el EDA:
1. Variable objetivo 'riesgo': 1 si 'final_result' es *Fail* o *Withdrawn*, 0 en caso contrario.
2. Doble ajuste en 'imd_band': se añade el símbolo de % que falta en el rango '10-20' y se sustituyen los nulos por la categoría 'Missing'.
3. Filtro de activos en día 27: activamos el filtro para excluir las 5040 matróculas con 'date_unregistration' =< 27. Nos quedamos con una muestra de modelado de **27.553 alumnos**, que supone un balance de clases de 44,2% riesgo = 1 y 55,8% riesgo = 0.

## 1. Features de VLE

Actividad del estudiante en la plataforma dentro de la ventana (días 0–27).

> **Decisión crítica:** según hemos visto en el EDA, la tabla'student_vle' tiene 1.43M (con keep=False) filas exactamente duplicadas (ver comentarios en la salida 1.1 para más detalle). Con esta situación es necesario aplicar 'drop_duplicates()' **antes** de cualquier agregación. Si se invierte el orden y se agrega primero, los clics quedan inflados hasta un 17%.

### 1.1 Deduplicación y filtro de ventana

In [3]:
# Paso 1: eliminar solo copias exactas (las 6 columnas iguales)
student_vle_clean = student_vle.drop_duplicates()
print(f"Filas originales:         {len(student_vle):,}")
print(f"Tras quitar exactas:      {len(student_vle_clean):,}")
print(f"Exactas eliminadas:       {len(student_vle) - len(student_vle_clean):,}")

# Verificación: claves con sum_click distinto entre filas (sesiones legítimas del mismo día)
variacion = (
    student_vle_clean
    .groupby(["code_module","code_presentation","id_student","id_site","date"])["sum_click"]
    .nunique()
)
print(f"\nClaves con sum_click distinto (sesiones del mismo día): {(variacion > 1).sum():,}")

# Paso 2: separar ventana oficial (0-27) de pre-curso (días negativos)
# Features de ventana y features pre-curso son señales distintas - no deben solaparse
vle_ventana  = student_vle_clean[
    (student_vle_clean["date"] >= 0) & (student_vle_clean["date"] <= 27)
].copy()
vle_precurso = student_vle_clean[student_vle_clean["date"] < 0].copy()

print(f"\nFilas en ventana oficial (date 0-27): {len(vle_ventana):,}")
print(f"Filas precurso (date < 0):            {len(vle_precurso):,}")

Filas originales:         10,655,280
Tras quitar exactas:      9,868,110
Exactas eliminadas:       787,170

Claves con sum_click distinto (sesiones del mismo día): 1,179,074

Filas en ventana oficial (date 0-27): 1,971,731
Filas precurso (date < 0):            657,160


**Comentarios:**

La secuencia de limpieza aplicada la establecimos ya en el EDA y se explica como decisión crítica antes del bloque de código de la celda anterior:

- **'drop_duplicates()' sobre las 6 columnas**: eliminamos 787.170 filas que son exactamente iguales debido a que son registros dentro de la misma sesión (esto es el mismo estudiante, el mismo día y con el mismo número de clics). Recordemos que en el EDA, el conteo ascendía a 1.429.783 porque usaba 'keep=False', lo que marca como duplicada toda fila del grupo, incluida la que se queda. Aquí contamos solo las que se borran realmente: filas_eliminadas = filas_implicadas - grupos_únicos_con_duplicado -> 787.170 = 1.429.783 - 642.613.

- **'groupby().sum()' sobre las 5 columnas clave** (sin 'sum_click'): consolidamos las 1.179.074 claves donde la misma persona tiene varias filas con el mismo material y día pero con variación del 'sum_click'. En este caso, sí son sesiones diferentes y el tratamiento más práctico es sumarlas para obtener el total de clics diario.

- **Tras la limpieza, la ventana de predicción contiene 1.971.731 filas.**

### 1.2 Clics totales y semanales

In [4]:
# Clave de agrupación a nivel estudiante
KEY = ["code_module", "code_presentation", "id_student"]

# Clics totales en ventana
total_clics = (
    vle_ventana
    .groupby(KEY)["sum_click"]
    .sum()
    .reset_index()
    .rename(columns={"sum_click": "total_clics"})
)

# Clics por semana (días 0-6, 7-13, 14-20, 21-27)
vle_ventana["semana"] = (vle_ventana["date"] // 7) + 1
clics_semana = (
    vle_ventana
    .groupby(KEY + ["semana"])["sum_click"]
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)
clics_semana.columns = (
    KEY + [f"clics_semana_{int(c)}" for c in clics_semana.columns[3:]]
)

# Verificación rápida
print(f"Estudiantes con actividad en ventana: {len(total_clics):,}")
print(f"\nDistribución de total_clics por grupo de riesgo (muestra):")
check = df_activos[KEY + ["riesgo"]].merge(total_clics, on=KEY, how="left").fillna(0)
print(check.groupby("riesgo")["total_clics"].median())

Estudiantes con actividad en ventana: 27,886

Distribución de total_clics por grupo de riesgo (muestra):
riesgo
0    222.0
1    114.5
Name: total_clics, dtype: float64


**Comentarios:**

Aquí realizamos los cálculos estableciendo dos familias de features para los clics:

- 'total_clics': la suma de todos los clics en el rango de 0 a 27 días. Aquí obtenemos una mediana de 222 clics para alumnos con riesgo = 0 y 114,5 para los alumnos con riesgo = 1. Más adelante veremos el peso que aporta esta variable en el modelo.
- 'clics_semana_1' a 'clics_semana_4': extraemos el volumen de clics durante cada semana, lo que permite que se capturen patrones temporales dentro de la propia ventana (si hay actividad inicial y después cae o viceversa, si tarda en arrancar pero luego aprieta). Debemos matizar que los alumnos sin ningún clic, constarán a 0 en todas estas columnas, tras el tratamiento en la celda de ensamblado con 'fillna'.

### 1.3 Días activos y regularidad

In [5]:
# Días distintos con actividad en ventana
dias_activo = (
    vle_ventana
    .groupby(KEY)["date"]
    .nunique()
    .reset_index()
    .rename(columns={"date": "dias_activo"})
)

# Regularidad: proporción de días con actividad sobre la ventana total (28 días)
dias_activo["regularidad"] = dias_activo["dias_activo"] / 28

# Verificación
check2 = df_activos[KEY + ["riesgo"]].merge(dias_activo, on=KEY, how="left").fillna(0)
print("Dias activos — mediana por grupo de riesgo:")
print(check2.groupby("riesgo")[["dias_activo", "regularidad"]].median())
print(f"\nEstudiantes sin ningún día activo en ventana: {(check2['dias_activo'] == 0).sum():,}")

Dias activos — mediana por grupo de riesgo:
        dias_activo  regularidad
riesgo                          
0              12.0     0.428571
1               7.0     0.250000

Estudiantes sin ningún día activo en ventana: 1,249


**Comentarios:**

- 'dias_activo': es la variable que cuenta el número de días distintos dentro de la ventana con al menos un clic.
- 'regularidad': normaliza el valor de 'dias_activo' dentro del total de la ventana (ratio 0-1)

La diferencia aquí es notable: los alumnos con riesgo = 0 están activos 12 de 28 días, mientras los de riesgo = 1 se quedan en 7 días de actividad, esto equivale a un 43% y 25% de regularidad respectivamente.

### 1.4 Tipos de actividad

> Teniendo claro que hay una importante variedad de actividades dentro del EVA, cruzamos 'vle.csv' por 'id_site' para obtener 'activity_type'. 'id_site' es una clave global, no se repite entre módulos distintos, así que el join no genera duplicados de más. Verificado en el EDA.

In [6]:
# Join con vle.csv para obtener activity_type
vle_ventana_tipo = vle_ventana.merge(
    vle[["id_site", "activity_type"]],
    on="id_site",
    how="left"
)

# Tipos distintos de actividad con los que interactuó cada estudiante
tipos_actividad = (
    vle_ventana_tipo
    .groupby(KEY)["activity_type"]
    .nunique()
    .reset_index()
    .rename(columns={"activity_type": "tipos_actividad"})
)

# Verificación
print(f"Activity types disponibles en vle.csv: {vle['activity_type'].nunique()}")
check3 = df_activos[KEY + ["riesgo"]].merge(tipos_actividad, on=KEY, how="left").fillna(0)
print("\nTipos de actividad — mediana por grupo de riesgo:")
print(check3.groupby("riesgo")["tipos_actividad"].median())
print(f"\nNulos tras el join (id_site sin match en vle.csv): {vle_ventana_tipo['activity_type'].isna().sum():,}")

Activity types disponibles en vle.csv: 20

Tipos de actividad — mediana por grupo de riesgo:
riesgo
0    7.0
1    6.0
Name: tipos_actividad, dtype: float64

Nulos tras el join (id_site sin match en vle.csv): 0


**Comentarios:**

A la vista de los resultados, no parece que el tipo de material vaya a ser determinante, si bien se mantiene como feature porque podría detectar perfiles de uso desequilibrados, como un estudiante que utilice únicamente un tipo de recurso.

### 1.5 Clics precurso

In [7]:
# Clics en días negativos (acceso anticipado antes del inicio del curso)
clics_precurso = (
    vle_precurso
    .groupby(KEY)["sum_click"]
    .sum()
    .reset_index()
    .rename(columns={"sum_click": "clics_precurso"})
)

# Verificación
check4 = df_activos[KEY + ["riesgo"]].merge(clics_precurso, on=KEY, how="left").fillna(0)
print("Clics precurso — mediana por grupo de riesgo:")
print(check4.groupby("riesgo")["clics_precurso"].median())
print(f"\nEstudiantes con actividad precurso: {len(clics_precurso):,}")
print(f"Estudiantes sin actividad precurso (fillna 0): {(check4['clics_precurso'] == 0).sum():,}")

Clics precurso — mediana por grupo de riesgo:
riesgo
0    39.0
1    14.0
Name: clics_precurso, dtype: float64

Estudiantes con actividad precurso: 23,809
Estudiantes sin actividad precurso (fillna 0): 5,476


**Comentarios:**

En este apartado, al contrario que en el anterior, sí que hay una distancia relevante entre grupos de riesgo y su volumen de actividad precurso (una mediana de 39 frente a 14), que es mayor incluso que dentro de la ventana. Se refuerza así lo que se anticipaba en el EDA respecto a que la separación entre grupos no emerge a lo largo del curso, sino que está ahí desde el inicio.

Nuevamente el colectivo de alumnos sin actividad precurso (un 20% de los activos) se marcarán como 0 en el procesado.

### 1.6 Ensamblar features VLE en un único DataFrame

> Consolidamos aquí todas las features del Espacio Virtual de Aprendizaje en un único dataframe con las 27.553 matrículas activas como base.

In [8]:
# Base: todos los estudiantes activos
df_vle_features = df_activos[KEY].copy()

# Joins sucesivos - left para conservar los 27.553 estudiantes
df_vle_features = (
    df_vle_features
    .merge(total_clics,    on=KEY, how="left")
    .merge(clics_semana,   on=KEY, how="left")
    .merge(dias_activo,    on=KEY, how="left")
    .merge(tipos_actividad, on=KEY, how="left")
    .merge(clics_precurso, on=KEY, how="left")
)

# Rellenar 0 en features de clics y días (estudiantes sin actividad)
# regularidad se recalcula desde dias_activo para no propagar NaN
cols_fill_0 = [c for c in df_vle_features.columns if c not in KEY + ["regularidad"]]
df_vle_features[cols_fill_0] = df_vle_features[cols_fill_0].fillna(0)
df_vle_features["regularidad"] = df_vle_features["dias_activo"] / 28

print(f"Shape df_vle_features: {df_vle_features.shape}")
print(f"\nNulos por columna:")
print(df_vle_features.isnull().sum())
print(f"\nFilas: {len(df_vle_features):,} (deben ser 27.553)")

Shape df_vle_features: (27553, 12)

Nulos por columna:
code_module          0
code_presentation    0
id_student           0
total_clics          0
clics_semana_1       0
clics_semana_2       0
clics_semana_3       0
clics_semana_4       0
dias_activo          0
regularidad          0
tipos_actividad      0
clics_precurso       0
dtype: int64

Filas: 27,553 (deben ser 27.553)


**Comentarios:**

Obtenemos 12 columnas resultantes y lo mas interesante es que todas tienen 0 valroes nulos, lo que significa que el tratamiento de estudiantes sin actividad con el fillna 0 es correcto y completo.

## 2. Features de Assessment

Evaluaciones entregadas dentro de la ventana: 'date_submitted <= 27', 'is_banked = 0', tipos TMA/CMA.
Módulos EEE y GGG no tienen evaluaciones en ventana - sus features quedarán en 0 o NaN como en casos anteriores.

### 2.1 Preparar assess_ventana

In [9]:
# Reconstruir assess_completo y assess_ventana desde cero (notebook autocontenido)
assess_completo = student_assess.merge(
    assessments[["id_assessment", "code_module", "code_presentation",
                 "assessment_type", "date", "weight"]],
    on="id_assessment"
)

# Filtro: entregas en ventana, nuevas (is_banked=0), TMA o CMA
assess_ventana = assess_completo[
    (assess_completo["date_submitted"] <= 27) &
    (assess_completo["is_banked"] == 0) &
    (assess_completo["assessment_type"].isin(["TMA", "CMA"]))
].copy()

print(f"Entregas en ventana (is_banked=0, TMA/CMA): {len(assess_ventana):,}")
print(f"Rango date_submitted: {assess_ventana['date_submitted'].min()} -> {assess_ventana['date_submitted'].max()}")

Entregas en ventana (is_banked=0, TMA/CMA): 23,439
Rango date_submitted: -11 -> 27


**Comentarios:**

Replicamos los 3 filtros aplicados en el EDA:
1. 'date_submitted =<27': limitamos la actividad al periodo establecido en la ventana de predicción.
2. 'is_banked = 0': excluimos las entregas importadas de semestres anteriores al no recoger comportamiento del estudiante dentro del periodo.
3. 'TMA y CMA': aunque el tipo 'Exam' no está presentes dentro de la ventana, incluimos explícitamente este filtro para que no entre ningún tipo imprevisto.

### 2.2 Features por estudiante: entregas y notas

In [10]:
# Módulos con evaluación en ventana (EEE y GGG no tienen -> sus features serán 0/NaN)
assess_en_ventana_distintas = assessments[assessments["date"] <= 27]
modulos_con_eval = assess_en_ventana_distintas["code_module"].unique()

# Agregar por estudiante
df_assess_features = (
    assess_ventana
    .groupby(KEY)
    .agg(
        n_entregas_ventana=("id_assessment", "count"),
        nota_media_ventana=("score",          "mean"),
        nota_min_ventana=  ("score",          "min")
    )
    .reset_index()
)

# entrego_algo: binaria derivada de n_entregas
df_assess_features["entrego_algo"] = (df_assess_features["n_entregas_ventana"] > 0).astype(int)

# Verificación
check_assess = df_activos[KEY + ["riesgo"]].merge(df_assess_features, on=KEY, how="left")

# Para EEE/GGG: n_entregas y entrego_algo -> 0; notas -> NaN (correcto, no hay evaluación)
check_assess["n_entregas_ventana"] = check_assess["n_entregas_ventana"].fillna(0)
check_assess["entrego_algo"]       = check_assess["entrego_algo"].fillna(0).astype(int)

print("Tasa de entrega por grupo de riesgo (denominador: módulos con eval en ventana):")
df_con_eval = check_assess[check_assess["code_module"].isin(modulos_con_eval)]
print(df_con_eval.groupby("riesgo")["entrego_algo"].mean().mul(100).round(1))

print("\nNotas en ventana por grupo de riesgo:")
print(check_assess.groupby("riesgo")[["nota_media_ventana","nota_min_ventana"]].mean().round(2))

print(f"\nNulos en nota_media_ventana: {check_assess['nota_media_ventana'].isna().sum():,}")
print("(NaN esperados: EEE/GGG + quien no entregó nada en módulos con eval)")

Tasa de entrega por grupo de riesgo (denominador: módulos con eval en ventana):
riesgo
0    96.7
1    75.2
Name: entrego_algo, dtype: float64

Notas en ventana por grupo de riesgo:
        nota_media_ventana  nota_min_ventana
riesgo                                      
0                    77.44             76.56
1                    66.31             65.57

Nulos en nota_media_ventana: 7,771
(NaN esperados: EEE/GGG + quien no entregó nada en módulos con eval)


**Comentarios:**

Recopilamos aquí cuatro features de evaluación por alumno:

1. 'n_entregas_ventana': nº de evaluaciones entregadas dentro de la ventana.
2. 'entrego_algo': variable binaria con valor 1 si n_entregas > 0. Aquí vemo como hay una variación de 21 puntos porcentuales entre los grupos de riesgo, lo que deja ver que no tener ninguna entrega en los primeros 27 días es un indicador de riesgo importante.
3. 'nota_media_ventana' y 'nota_min_ventana': estadísticos de las notas obtenidas, con una variación de 11 puntos en la media entre riesgo = 0 y riesgo = 1.

Los nulos recogidos en la nota media (7.771) provienen de dos grupos que ya conocíamos que no tienen nota: por un lado, los estudiantes de EEE y GGG (módulos sin evaluación en ventana) y por otro lado, quienes siendo estudiantes de módulos con evaluación, no entregaron nada.

In [11]:
# Consolidar: n_entregas y entrego_algo -> 0 si sin entrega; notas -> NaN 
df_assess_features["n_entregas_ventana"] = df_assess_features["n_entregas_ventana"].fillna(0)
df_assess_features["entrego_algo"]       = df_assess_features["entrego_algo"].fillna(0).astype(int)
# nota_media_ventana y nota_min_ventana se dejan como NaN intencionadamente

print(f"Shape df_assess_features: {df_assess_features.shape}")
print(f"Nulos por columna:\n{df_assess_features.isnull().sum()}")

Shape df_assess_features: (20185, 7)
Nulos por columna:
code_module            0
code_presentation      0
id_student             0
n_entregas_ventana     0
nota_media_ventana    13
nota_min_ventana      13
entrego_algo           0
dtype: int64


**Comentarios:**

Vemos aquí el total de filas de 'df_assess_features' después de pasar por el join y el fillna, de lo que obtenemos 20.185 alumnos con intentos de entrega en la ventana ('assess_ventana').

Los 13 intentos de entrega sin calificación, los consolidamos aquí NaN y 0 de manera expresa para que cada modelo en las iteraciones del H3 decida su tratamiento (imputación o gestión nativa).

## 3. Ensamblado: 'df_modelo'

> Realizamos en esta sección en ensamble del df final para el modelo, tomando como base: 'df_activos' (27.553 filas, todas las demográficas ya limpias).
> Usamos left join con features VLE -> left join con features Assessment.
> Por último, rellenamos con fillna: 0 en clics/días para sin actividad; NaN en notas para EEE/GGG y quien no entregó.

### 3.1 Join

In [12]:
# Base: todos los estudiantes activos con sus demográficas
df_modelo = df_activos.drop(columns=["date_unregistration"]).copy()

# Join con features VLE (left: conserva los 27.553)
df_modelo = df_modelo.merge(df_vle_features, on=KEY, how="left")

# Join con features de assessment
df_modelo = df_modelo.merge(df_assess_features, on=KEY, how="left")

# Fillna tras el join: 0 para conteos, NaN intencional para notas
df_modelo["n_entregas_ventana"] = df_modelo["n_entregas_ventana"].fillna(0)
df_modelo["entrego_algo"]       = df_modelo["entrego_algo"].fillna(0).astype(int)
# nota_media_ventana y nota_min_ventana se quedan NaN para EEE/GGG y no entregaron

print(f"Shape df_modelo: {df_modelo.shape}")
print(f"\nNulos por columna:")
print(df_modelo.isnull().sum()[df_modelo.isnull().sum() > 0])
print(f"\nBalance de clases: riesgo = 1 -> {df_modelo['riesgo'].mean():.3f}")

Shape df_modelo: (27553, 26)

Nulos por columna:
nota_media_ventana    7771
nota_min_ventana      7771
dtype: int64

Balance de clases: riesgo = 1 -> 0.442


**Comentarios:**

Verificamos en la salida que los NaN en las notas son correctos, 7.771 que corresponden a los alumnos de EEE, GGG o sin entrega en módulos con evaluación. Además se mantiene el balance de clases en 44,2% de riesgo.

### 3.2 Verificación de leakage

Comenzamos quitando expresamente 'final_result' del dataframe ya que define explícitamente la variable objetivo 'riesgo', con lo que mantenerlo sería un leakage directo.

In [13]:
df_modelo = df_modelo.drop(columns=["final_result"])
print(f"Columnas tras eliminar final_result: {df_modelo.shape[1]}")
print(df_modelo.columns.tolist())

Columnas tras eliminar final_result: 25
['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'riesgo', 'total_clics', 'clics_semana_1', 'clics_semana_2', 'clics_semana_3', 'clics_semana_4', 'dias_activo', 'regularidad', 'tipos_actividad', 'clics_precurso', 'n_entregas_ventana', 'nota_media_ventana', 'nota_min_ventana', 'entrego_algo']


**Comentarios:**

Se consolida el dataset en 25 columnas con la siguiente distribución:
- 3 de identificación ('code_module', 'code_presentation' y 'id_student').
- 1 objetivo ('riesgo').
- 21 features.

### 3.3 Shape, nulos y balance final

Antes de continuar con el proceso de exportación, debemos verificar varias cuestiones adicionales:

- 'date_unregistration' no está en df_modelo.
- Ninguna feature de VLE usa datos de date > 27.
- Ninguna feature de assessment usa date_submitted > 27 ni is_banked = 1.
- El target ('riesgo') no está correlacionado trivialmente con ninguna feature por construcción.

In [14]:
# 1. Ninguna columna que defina o filtre el target
assert "final_result" not in df_modelo.columns, "LEAKAGE: final_result presente"
assert "date_unregistration" not in df_modelo.columns, "LEAKAGE: date_unregistration presente"

# 2. El target está y tiene el balance esperado
assert "riesgo" in df_modelo.columns
assert abs(df_modelo["riesgo"].mean() - 0.442) < 0.01, "Balance de clases inesperado"

# 3. Filas y columnas
assert len(df_modelo) == 27553, f"Filas incorrectas: {len(df_modelo)}"
assert df_modelo.shape[1] == 25, f"Columnas incorrectas: {df_modelo.shape[1]}"

# 4. Solo NaN permitidos: notas (EEE/GGG y no entregaron)
nulos = df_modelo.isnull().sum()
cols_con_nulos = nulos[nulos > 0]
cols_permitidas_con_nulos = {"nota_media_ventana", "nota_min_ventana"}
cols_inesperadas = set(cols_con_nulos.index) - cols_permitidas_con_nulos
assert len(cols_inesperadas) == 0, f"Nulos inesperados en: {cols_inesperadas}"

print("Todos los checks de leakage pasados.")
print(f"Shape final: {df_modelo.shape}")
print(f"NaN permitidos — nota_media: {df_modelo['nota_media_ventana'].isna().sum():,} | nota_min: {df_modelo['nota_min_ventana'].isna().sum():,}")

Todos los checks de leakage pasados.
Shape final: (27553, 25)
NaN permitidos — nota_media: 7,771 | nota_min: 7,771


**Comentarios:**

Ejecutamos cuatro asserts de seguridad para verificar los 4 puntos establecidos en la celda previa. Resultados:

1. 'final_result' y 'date_unregistration' no están presentes.
2. 'riesgo' está presente con el balance dentro del margen esperado.
3. El shape es exacto: 27.553 x 25 columnas.
4. Valores NaN permitidos solo en 'nota_media_ventana' y 'nota_min_ventana'.

## 4. Exportar 'df_modelo'

In [15]:
OUTPUT_PATH = "../datos/df_modelo.csv"
df_modelo.to_csv(OUTPUT_PATH, index=False)
print(f"df_modelo exportado: {OUTPUT_PATH}")
print(f"Shape: {df_modelo.shape}")

df_modelo exportado: ../datos/df_modelo.csv
Shape: (27553, 25)


**Comentarios:**

El dataset para el modelado queda cerrado y exportado a 'datos/df_modelo.csv', es el único archivo de entrada que necesita el notebook del modelo (H3).